# 02 — Data Cleaning

**Input:** `data/raw/monthly_ev_sales.csv`
**Outputs:**
- `data/processed/registrations.parquet` — clean long-format training table
- `data/processed/aggregates.parquet` — aggregate rows for share calculations
- `data/processed/series_metadata.parquet` — per-series coverage flags
- `data/sample/processed_sample.csv` — small committed sample

**Contract source:** `docs/dataset_sources.md`

**Rules applied:**
1. Parse dates.
2. Convert membership flags to bool.
3. Split training vs aggregate rows.
4. Exclude Region rows from training.
5. Exclude structurally-absent series (all zero).
6. Reindex to complete monthly grid per series; structural zeros → 0, gaps → flagged.
7. Save parquet.

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path("../data/raw/monthly_ev_sales.csv")
PROCESSED_DIR = Path("../data/processed")
SAMPLE_DIR = Path("../data/sample")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

MEMBERSHIP_COLS = ["EU member", "OECD member", "G20 member", "G7 member", "ASEAN member"]

## Load & Clean Data Types

In [25]:
df = pd.read_csv(RAW_PATH)
df["Date"] = pd.to_datetime(df["Date"], errors="raise")

# Membership flags: object -> bool
for col in MEMBERSHIP_COLS:
    df[col] = df[col].astype(str).str.lower().isin(["true", "1", "yes"])

print("Membership dtypes:")
print(df[MEMBERSHIP_COLS].dtypes)
print("\nValue counts for EU member:")
print(df["EU member"].value_counts())

Membership dtypes:
EU member       bool
OECD member     bool
G20 member      bool
G7 member       bool
ASEAN member    bool
dtype: object

Value counts for EU member:
EU member
False    25438
True     21432
Name: count, dtype: int64


## Split training vs aggregates and exclude regions

In [26]:
train_raw = df[df["Is aggregate drivetrain"] == False].copy()
agg_raw = df[df["Is aggregate drivetrain"] == True].copy()

# Drop Region rows from training — never train on aggregates
train_countries = train_raw[train_raw["Area type"] == "Country or economy"].copy()

print(f"Rows: train_raw={len(train_raw)}, agg_raw={len(agg_raw)}, train_countries={len(train_countries)}")
print(f"Countries: {train_countries['Area'].nunique()}")
print(f"Regions excluded from training: {train_raw['Area type'].unique().tolist()}")

Rows: train_raw=34092, agg_raw=12778, train_countries=33432
Countries: 59
Regions excluded from training: ['Country or economy', 'Region']


## Identify structural zeros

In [27]:
series_stats = (
    train_countries.groupby(["ISO 3 code", "Area", "Drivetrain"])
    .agg(
        n_obs=("Registrations", "size"),
        n_nonzero=("Registrations", lambda s: (s > 0).sum()),
        first_date=("Date", "min"),
        last_date=("Date", "max"),
    )
    .reset_index()
)

series_stats["is_structural_zero"] = series_stats["n_nonzero"] == 0
series_stats["has_min_history"] = series_stats["n_obs"] >= 12

print(f"Total series:                 {len(series_stats)}")
print(f"Structural-zero series:       {series_stats['is_structural_zero'].sum()}")
print(f"Series with >=12 obs:         {series_stats['has_min_history'].sum()}")

# Save for reference
series_stats.to_parquet(PROCESSED_DIR / "series_metadata.parquet", index=False)

Total series:                 316
Structural-zero series:       12
Series with >=12 obs:         316


## Drop structural zeros from training

In [28]:
valid_series = series_stats.loc[~series_stats["is_structural_zero"], ["ISO 3 code", "Drivetrain"]]

train_clean = train_countries.merge(
    valid_series.assign(_valid=True),
    on=["ISO 3 code", "Drivetrain"],
    how="inner",
)

print(f"Rows before structural-zero drop: {len(train_countries)}")
print(f"Rows after drop:                  {len(train_clean)}")
print(f"Dropped:                          {len(train_countries) - len(train_clean)}")

Rows before structural-zero drop: 33432
Rows after drop:                  33058
Dropped:                          374


##  Reindex to complete monthly grid

In [29]:

def reindex_series(group: pd.DataFrame) -> pd.DataFrame:
    """Reindex one country-drivetrain series to a complete monthly grid."""

    iso3, drivetrain = group.name

    group = group.copy()
    group = group.sort_values("Date").set_index("Date")

    full_idx = pd.date_range(
        start=group.index.min(),
        end=group.index.max(),
        freq="MS"
    )

    group = group.reindex(full_idx)
    group.index.name = "Date"

    # Restore grouping keys after reindexing
    group["ISO 3 code"] = iso3
    group["Drivetrain"] = drivetrain

    # Identify months inserted during reindexing
    group["was_missing"] = group["Registrations"].isna()

    # Keep missing observations distinguishable from actual zero registrations
    return group.reset_index()


reindexed = (
    train_clean
    .groupby(["ISO 3 code", "Drivetrain"], group_keys=False)
    .apply(reindex_series)
    .reset_index(drop=True)
)

print(f"Rows before reindex: {len(train_clean)}")
print(f"Rows after reindex:  {len(reindexed)}")
print(f"Inserted missing months: {reindexed['was_missing'].sum()}")
print("Available columns:", reindexed.columns.tolist())


Rows before reindex: 33058
Rows after reindex:  33064
Inserted missing months: 6
Available columns: ['Date', 'Area', 'Area type', 'Data frequency', 'Is aggregate drivetrain', 'Registrations', 'Registrations YoY change', 'Registrations YoY change (%)', 'Share of registrations (%)', 'Year-to-date registrations', 'Year-to-date share of registrations (%)', 'Rolling 3-month registrations', 'Rolling 3-month share of registrations (%)', 'Rolling 12-month registrations', 'Rolling 12-month share of registrations (%)', 'Continent', 'Ember region', 'EU member', 'OECD member', 'G20 member', 'G7 member', 'ASEAN member', '_valid', 'ISO 3 code', 'Drivetrain', 'was_missing']


## Restore static metadata to reindexed rows

In [31]:

static_cols = [
    "Area",
    "Area type",
    "Continent",
    "Ember region"
] + MEMBERSHIP_COLS

# Create one metadata record per country-drivetrain combination
static_lookup = (
    train_clean[
        ["ISO 3 code", "Drivetrain"] + static_cols
    ]
    .drop_duplicates(subset=["ISO 3 code", "Drivetrain"])
)

# Remove existing metadata columns to prevent _x and _y suffixes
reindexed = reindexed.drop(
    columns=static_cols,
    errors="ignore"
)

# Restore metadata, including for newly inserted months
reindexed = reindexed.merge(
    static_lookup,
    on=["ISO 3 code", "Drivetrain"],
    how="left",
    validate="many_to_one"
)

# Restore boolean dtypes
for col in MEMBERSHIP_COLS:
    reindexed[col] = reindexed[col].astype(bool)

# Verification
print("Missing metadata values:")
print(reindexed[static_cols].isna().sum())

print("\nMembership column dtypes:")
print(reindexed[MEMBERSHIP_COLS].dtypes)

print("\nFinal shape:", reindexed.shape)
print("\nDuplicate country-drivetrain-date keys:",
      reindexed.duplicated(
          ["ISO 3 code", "Drivetrain", "Date"]
      ).sum())


Missing metadata values:
Area            0
Area type       0
Continent       0
Ember region    0
EU member       0
OECD member     0
G20 member      0
G7 member       0
ASEAN member    0
dtype: int64

Membership column dtypes:
EU member       bool
OECD member     bool
G20 member      bool
G7 member       bool
ASEAN member    bool
dtype: object

Final shape: (33064, 44)

Duplicate country-drivetrain-date keys: 0


## Build Final Canonical Table

In [35]:
final_cols = [
    "ISO 3 code", "Area", "Area type", "Drivetrain",
    "Date", "Registrations",
    "Continent", "Ember region",
] + MEMBERSHIP_COLS + ["was_missing"]

registrations = (
    reindexed[final_cols]
    .rename(columns={
        "ISO 3 code": "iso3",
        "Area": "area",
        "Area type": "area_type",
        "Drivetrain": "drivetrain",
        "Date": "date",
        "Registrations": "registrations",
        "Continent": "continent",
        "Ember region": "ember_region",
        "EU member": "is_eu",
        "OECD member": "is_oecd",
        "G20 member": "is_g20",
        "G7 member": "is_g7",
        "ASEAN member": "is_asean",
        "was_missing": "was_missing",
    })
    .sort_values(["iso3", "drivetrain", "date"])
    .reset_index(drop=True)
)

print(registrations.dtypes)
print(registrations.head())

print(f"\nShape: {registrations.shape}")
print(f"Date range: {registrations['date'].min().date()} to {registrations['date'].max().date()}")
print(f"Countries: {registrations['iso3'].nunique()}")
print(f"Series: {registrations.groupby(['iso3', 'drivetrain']).ngroups}")
print(f"Missing registrations: {registrations['registrations'].isna().sum()}")
print(f"Flagged missing rows: {registrations['was_missing'].sum()}")

iso3                        str
area                        str
area_type                   str
drivetrain                  str
date             datetime64[us]
registrations           float64
continent                   str
ember_region                str
is_eu                      bool
is_oecd                    bool
is_g20                     bool
is_g7                      bool
is_asean                   bool
was_missing                bool
dtype: object
  iso3     area           area_type        drivetrain       date  \
0  ALB  Albania  Country or economy  Battery electric 2019-01-01   
1  ALB  Albania  Country or economy  Battery electric 2019-02-01   
2  ALB  Albania  Country or economy  Battery electric 2019-03-01   
3  ALB  Albania  Country or economy  Battery electric 2019-04-01   
4  ALB  Albania  Country or economy  Battery electric 2019-05-01   

   registrations continent ember_region  is_eu  is_oecd  is_g20  is_g7  \
0            1.0    Europe       Europe  False    False

## Save Processed Artifacts

In [36]:
registrations.to_parquet(PROCESSED_DIR / "registrations.parquet", index=False)

# Also save aggregates (World, EU, etc.) for share calculations in later phases
agg_cols = ["ISO 3 code", "Area", "Area type", "Drivetrain", "Date",
            "Registrations", "Continent", "Ember region"]
aggregates = (
    agg_raw[agg_cols]
    .rename(columns={"ISO 3 code": "iso3", "Area": "area", "Area type": "area_type",
                     "Drivetrain": "drivetrain", "Date": "date",
                     "Registrations": "registrations",
                     "Continent": "continent", "Ember region": "ember_region"})
    .sort_values(["iso3", "drivetrain", "date"])
    .reset_index(drop=True)
)
aggregates.to_parquet(PROCESSED_DIR / "aggregates.parquet", index=False)

# Small sample for committing
sample = registrations.sample(n=min(2000, len(registrations)), random_state=42)
sample.to_csv(SAMPLE_DIR / "processed_sample.csv", index=False)

print(f"registrations.parquet: {len(registrations)} rows")
print(f"aggregates.parquet:    {len(aggregates)} rows")
print(f"processed_sample.csv:  {len(sample)} rows (committed)")

registrations.parquet: 33064 rows
aggregates.parquet:    12778 rows
processed_sample.csv:  2000 rows (committed)


## Sanity checks

In [37]:
# 1. Every series must have contiguous monthly dates
def check_contiguity(g):
    diffs = g["date"].diff().dropna().dt.days
    return ((diffs >= 28) & (diffs <= 31)).all()

contiguous = registrations.groupby(
    ["iso3", "drivetrain"]
).apply(check_contiguity)

print(f"Series with contiguous months: {contiguous.sum()}/{len(contiguous)}")
assert contiguous.all(), "Non-contiguous series found — investigate."


# 2. Every missing registration must be explicitly flagged
missing_target = registrations["registrations"].isna()

assert registrations["was_missing"].notna().all(), \
    "Missing-value flags contain NaN."

assert registrations.loc[missing_target, "was_missing"].all(), \
    "Unflagged NaN registrations found."

assert not registrations.loc[~registrations["was_missing"], "registrations"].isna().any(), \
    "NaN registrations exist without a missing-data flag."

print(f"Missing registrations: {missing_target.sum()}")
print(f"Flagged missing rows: {registrations['was_missing'].sum()}")


# 3. No duplicate keys
duplicate_count = registrations.duplicated(
    ["iso3", "drivetrain", "date"]
).sum()

print(f"Duplicate keys: {duplicate_count}")

assert duplicate_count == 0, "Duplicate records found."


print("\nAll sanity checks passed.")

Series with contiguous months: 304/304
Missing registrations: 6
Flagged missing rows: 6
Duplicate keys: 0

All sanity checks passed.


## Findings

## Phase 1 Cleaning — Findings

### 1. Dataset Overview
- **Input dataset:** Ember Monthly EV Sales Data
- **Input rows:** 46,870
- **Output rows:** 33,064
- **Countries retained:** 57
- **Country–drivetrain series retained:** 304
- **Date range:** Varies by country and drivetrain; each series retains its own available date range.

### 2. Data Cleaning
- Parsed the `Date` column into a datetime format.
- Converted EU, OECD, G20, G7 and ASEAN membership flags into Boolean values.
- Separated aggregate drivetrain records from country-level training records.
- Excluded regional aggregate records from the training dataset.
- Identified and excluded 12 structurally-zero country–drivetrain series.
- Reindexed retained series to complete monthly date sequences.
- Preserved missing registration values as NaN and marked them using the `was_missing` flag rather than assuming that unreported months represent zero registrations.

### 3. Data Quality Results
- **Rows after structural-zero removal:** 33,058
- **Rows after monthly reindexing:** 33,064
- **Additional monthly records inserted:** 6
- **Missing registration values:** 6
- **Flagged missing observations:** 6
- **Series with contiguous monthly dates:** 304/304 (100%)
- **Duplicate country–drivetrain–date keys:** 0
- **Membership columns:** All converted to Boolean.

### 4. Validation
All implemented sanity checks passed:
- Monthly date continuity verified for every retained series.
- All missing registration values have corresponding missing-data flags.
- No unflagged missing registration values found.
- No duplicate country–drivetrain–date combinations found.

### 5. Output Artifacts
- `data/processed/registrations.parquet` — cleaned country-level registration data for modelling.
- `data/processed/aggregates.parquet` — aggregate records retained separately for market-share calculations.
- `data/processed/series_metadata.parquet` — country–drivetrain series coverage and structural-zero metadata.
- `data/sample/processed_sample.csv` — sample of processed records for repository demonstration.

### 6. Conclusion
The dataset has successfully passed the initial cleaning and structural validation stage. It is now organized into a consistent country–drivetrain monthly time-series format, with missing observations explicitly tracked and aggregate records kept separate from country-level training data.

**Note:** The six missing observations will require an explicit missing-data treatment decision before model training.